In [ ]:
from prometheus_pandas import query
import datetime as dt
from zoneinfo import ZoneInfo
import pandas as pd
import matplotlib.pyplot as plt

local_address = "nova-16.lyon.grid5000.fr"
local_port = "30090"
p = query.Prometheus(f"http://admin:prom-operator@{local_address}:{local_port}")    

local_tz = ZoneInfo("Europe/Paris")

def execute_query_range(experiment, metric_query, step, output_filename, plot=False):
    tput_series = []

    print("\nBeginning request execution")
    for label, config in experiment.items():    
        df_metric = p.query_range(
            query=metric_query,
            start=config['start'].isoformat(),
            end=config['end'].isoformat(),
            step=step
        )

        tput_series.append(df_metric)

    if tput_series:
        combined_df = pd.concat(tput_series, axis=1)
        combined_df.to_csv(output_filename, index_label='Time')
        print(f"\nFile saved : '{output_filename}'")
        if plot:
            plt.figure(figsize=(12, 8))
            ax = combined_df.boxplot(grid=True, patch_artist=True)
            plt.title(metric_query)
            plt.show()
    else:
        print("\nRequest Error")

In [ ]:
exp_q1 = {
    'rocks': {
        'start': dt.datetime(2025, 7, 16, 11, 1, 0, tzinfo=local_tz),
        'end': dt.datetime(2025, 7, 16, 11, 5, 0, tzinfo=local_tz)
    },
    'forst': {
        'start': dt.datetime(2025, 7, 16, 11, 10, 0, tzinfo=local_tz),
        'end': dt.datetime(2025, 7, 16, 11, 15, 0, tzinfo=local_tz)
    },
    'forst_no_cache': {
        'start': dt.datetime(2025, 7, 16, 11, 20, 0, tzinfo=local_tz),
        'end': dt.datetime(2025, 7, 16, 11, 25, 0, tzinfo=local_tz)
    }
}

In [ ]:
metric_query = 'sum(flink_taskmanager_job_task_numRecordsInPerSecond{component="taskmanager"})'
step = "5s"
output_filename = 'exp2a_q1_tput.csv'

execute_query_range(exp_q1, metric_query, step, output_filename)